# Step 3: Multi-Agent Patterns — The Crew Chief

**Goal:** Build a working multi-agent system where one LLM (the supervisor) routes tasks to specialist LLMs (researcher, writer).

**What you'll learn:**
- **LLMs inside nodes** — replacing if/else with actual AI calls
- **Message-based state** — the standard communication pattern for multi-agent systems
- **The supervisor pattern** — one LLM routing to specialists, with workers reporting back

**Prerequisites:** Steps 1-2 completed, `pip install langchain-anthropic`, Anthropic API key set as `ANTHROPIC_API_KEY`

**Time:** ~60 minutes

---

### The Conceptual Leap

In Steps 1-2, nodes used keyword matching and threshold checks. That was deliberate: **learn the plumbing before turning on the water.**

Now we turn on the water. The graph topology stays the same — nodes, edges, conditional routing. But instead of `if "enemy" in report`, we have `llm.invoke()`. The LLM is the person sitting in the staff section seat. LangGraph doesn't care whether it's a keyword matcher or Claude. It just knows: the node runs, reads state, and returns updates.

> **Same interface, different brain.** That's the architectural insight that makes LangGraph production-ready.

## Key Concept: Messages as State

In Steps 1-2, state had simple fields (`query`, `findings`). Multi-agent systems use **messages** as the primary state:

```python
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add]   # Conversation history (APPENDS)
    next_agent: str                                # Routing decision
```

Why messages? Because **agents communicate by appending to a shared conversation**:
- The researcher writes: *"[Researcher]: I found these three documents."*
- The writer reads that and writes: *"[Writer]: Here's my synthesis."*
- The supervisor reads everything and decides what's next.

The reducer (`add`) ensures messages **accumulate** — by Phase 4, the writer sees the original query AND all research findings. No manual passing required.

### Key Concept: One LLM, Many Hats

Different `SystemMessage` prompts create different specialist personas from the **same underlying model**:

| Node | SystemMessage | Persona |
|---|---|---|
| Supervisor | "You are a coordinator..." | Routes tasks, decides when done |
| Researcher | "You are a researcher..." | Finds facts, cites reasoning |
| Writer | "You are a writer..." | Synthesizes findings into answers |

In [ ]:
from typing import Annotated, TypedDict, Literal
from operator import add
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# ─── THE LLM ────────────────────────────────────────────────────────────
# Single instance, shared by all nodes. Each node gives it a different
# SystemMessage = different specialist persona.
llm = ChatAnthropic(model="claude-sonnet-4-20250514")

# ─── MESSAGE-BASED STATE ────────────────────────────────────────────────
# messages uses a reducer (add) → each node APPENDS to the conversation.
# next_agent is overwrite → supervisor sets it fresh each time.

class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add]   # Full conversation (APPENDS)
    next_agent: str                                # Who goes next (overwrite)
    final_answer: str                              # Finished output (overwrite)

print("LLM initialized and AgentState defined")
print("State has 1 reducer field (messages) and 2 overwrite fields")

## The Supervisor Node

The supervisor is the routing brain. It reads the full conversation and asks the LLM: "Who should handle this next?"

**Key difference from Step 1's router:** Instead of `if/else` logic, an LLM makes the routing decision. The LLM can reason about nuance that rules can't capture.

The supervisor's output is constrained to three words: `"researcher"`, `"writer"`, or `"FINISH"`. This makes parsing reliable. In production, you'd use structured output for even more reliability.

In [ ]:
def supervisor(state: AgentState) -> dict:
    """
    NODE: supervisor
    Role: The Director — reads conversation, decides who goes next.
    Reads: messages (full history)
    Writes: next_agent (routing decision)
    
    The LLM sees the ENTIRE conversation (thanks to the reducer)
    and decides: need more research? ready for writing? all done?
    """
    messages = state["messages"]

    supervisor_prompt = [
        SystemMessage(content="""You are a supervisor coordinating a research team.
Your team has two specialists:
  - "researcher": Finds and retrieves information. Use when you need facts or data.
  - "writer": Synthesizes information into a clear answer. Use when you have enough facts.

Based on the conversation so far, decide who should act next.
If the research is complete and the writer has produced a good answer, respond with "FINISH".

Respond with ONLY one word: "researcher", "writer", or "FINISH".
No explanation, no punctuation — just the single word."""),
    ]
    supervisor_prompt.extend(messages)

    response = llm.invoke(supervisor_prompt)
    decision = response.content.strip().lower().strip('"').strip("'")
    print(f"  [supervisor] Decision: {decision}")

    return {"next_agent": decision}

print("Supervisor node defined")

## The Worker Nodes

Workers are specialists. Each gets a different `SystemMessage` that defines their expertise. They:
1. Read the conversation history (via state)
2. Do their specialized work (via LLM call)
3. Append their output as a tagged message (via reducer)

**Why `HumanMessage` instead of `AIMessage`?** From the *next* agent's perspective, a colleague's output is **input** (like someone handing them notes), not their own prior response. The `[Researcher]:` prefix helps agents identify who said what.

In [ ]:
def researcher(state: AgentState) -> dict:
    """
    NODE: researcher
    Role: Specialist who finds information.
    Reads: messages (to understand context)
    Writes: messages (APPENDS findings as tagged HumanMessage)
    """
    messages = state["messages"]

    research_prompt = [
        SystemMessage(content="""You are an expert researcher. Your job is to find
relevant information based on the conversation. Be specific and cite your reasoning.
Keep your response focused and under 3 sentences — you're feeding a writer, not
writing the final answer yourself."""),
    ]
    research_prompt.extend(messages)

    response = llm.invoke(research_prompt)
    print(f"  [researcher] Found: {response.content[:80]}...")

    return {
        "messages": [HumanMessage(content=f"[Researcher]: {response.content}")],
    }


def writer(state: AgentState) -> dict:
    """
    NODE: writer
    Role: Synthesizes findings into a polished answer.
    Reads: messages (including researcher's findings)
    Writes: messages (APPENDS draft), final_answer
    
    The writer sees EVERYTHING: original query, all findings, any prior drafts.
    """
    messages = state["messages"]

    writer_prompt = [
        SystemMessage(content="""You are an expert writer. Synthesize the research
findings in the conversation into a clear, concise answer for the user.
If you don't have enough information, say what's missing.
Keep your answer under 4 sentences."""),
    ]
    writer_prompt.extend(messages)

    response = llm.invoke(writer_prompt)
    print(f"  [writer] Wrote: {response.content[:80]}...")

    return {
        "messages": [HumanMessage(content=f"[Writer]: {response.content}")],
        "final_answer": response.content,
    }


def route_to_agent(state: AgentState) -> str:
    """ROUTER: Translates supervisor's decision into a graph edge."""
    next_agent = state.get("next_agent", "").lower()
    if next_agent == "finish":
        return "end"
    elif next_agent == "writer":
        return "writer"
    return "researcher"

print("Researcher, Writer, and Router defined")

## Wiring the Supervisor Loop

The topology creates a natural loop:

```
START → supervisor → [researcher|writer|END]
              ↑              |          |
              └──────────────┘──────────┘
```

Workers **always report back to the supervisor**. The supervisor re-evaluates: "Do we need another specialist, or are we done?" The loop exits when the supervisor says `"FINISH"`.

This is **agent-driven iteration** — no hardcoded loop count. The LLM decides when the job is done.

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("supervisor", supervisor)
graph.add_node("researcher", researcher)
graph.add_node("writer", writer)

graph.add_edge(START, "supervisor")

graph.add_conditional_edges(
    "supervisor",
    route_to_agent,
    {
        "researcher": "researcher",
        "writer": "writer",
        "end": END,
    },
)

# Workers → back to supervisor (the loop)
graph.add_edge("researcher", "supervisor")
graph.add_edge("writer", "supervisor")

memory = MemorySaver()
app = graph.compile(checkpointer=memory)

print("Multi-agent graph compiled!")

## Running the Multi-Agent System

Watch the supervisor-worker loop in action. The supervisor will:
1. Route to the researcher first (needs facts)
2. Re-evaluate after research (has enough → route to writer)
3. Re-evaluate after writing (good answer → FINISH)

**What to watch for:** The same Claude model produces different responses based on the SystemMessage. That's the "one LLM, many hats" pattern.

In [ ]:
config = {"configurable": {"thread_id": "multi-agent-session-1"}}

print("=" * 60)
print("MULTI-AGENT RESEARCH SYSTEM")
print("=" * 60)
print()

result = app.invoke(
    {
        "messages": [
            HumanMessage(content="What is LangGraph and how does it compare to plain LangChain?")
        ],
        "next_agent": "",
        "final_answer": "",
    },
    config=config,
)

print()
print("=" * 60)
print("FINAL ANSWER")
print("=" * 60)
print(result["final_answer"])

In [ ]:
print("=" * 60)
print("FULL AGENT CONVERSATION (your audit trail)")
print("=" * 60)
for i, msg in enumerate(result["messages"]):
    role = "USER" if isinstance(msg, HumanMessage) else "AI"
    content_preview = msg.content[:120]
    print(f"\n  [{i}] {role}: {content_preview}...")

## Key Takeaways

| Concept | Implementation | Why It Matters |
|---|---|---|
| **LLMs in nodes** | `llm.invoke([SystemMsg, HumanMsg])` | Same graph topology, AI-powered decisions |
| **Message state** | `Annotated[list[BaseMessage], add]` | Agents communicate via shared conversation |
| **Supervisor pattern** | One LLM routes, specialists execute | Dynamic multi-step workflows |
| **One LLM, many hats** | Different `SystemMessage` per node | Same model becomes different specialists |
| **Agent-driven loops** | Supervisor decides when to FINISH | No hardcoded iteration counts |

**The runtime execution flow:**
1. User query → Supervisor routes → Worker executes → Back to supervisor
2. Supervisor re-evaluates → Another worker or FINISH
3. Full conversation preserved via reducer at every step

> **Interview signal:** "The orchestration is generic — the supervisor pattern works for any domain. The specialization lives in the SystemMessage prompts. You can add a new specialist by adding one node and one edge, without touching any other node."

---
*Next: [04_rag_pipeline.ipynb](./04_rag_pipeline.ipynb) — Production RAG with quality gates and feedback loops*